# 02 — Model training & validation (AChE / Alzheimer's potency predictor)

Run in Google Colab, right after notebook 1. Upload `cleaned_data.csv` when prompted (or point it at the same Drive folder).

**What this notebook does:**
1. Loads the cleaned, descriptor-enriched dataset (default target: acetylcholinesterase, `CHEMBL220`).
2. Trains a RandomForest QSAR model predicting **pIC50** (potency) from molecular descriptors.
3. Validates it with a held-out test set + 5-fold cross-validation.
4. Reports R², RMSE, MAE and feature importance.
5. Saves the trained model (`model.pkl`) and the descriptor list (`feature_names.json`) — both are what the Streamlit app loads.

If you swapped to a different Alzheimer's target in notebook 1 (BACE1, GSK-3β, BChE), nothing below needs to change — it trains on whatever `cleaned_data.csv` contains.


In [ ]:
!pip install -q rdkit scikit-learn joblib


In [ ]:
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


In [ ]:
# Upload cleaned_data.csv from notebook 1 (skip this cell if it's already in the Colab session)
from google.colab import files
uploaded = files.upload()  # choose cleaned_data.csv


In [ ]:
df = pd.read_csv("cleaned_data.csv")
FEATURES = ["MolWt", "LogP", "TPSA", "HBD", "HBA", "RotatableBonds", "RingCount", "AromaticRings"]
TARGET = "pIC50"

X = df[FEATURES].values
y = df[TARGET].values
print(f"Training on {len(df)} compounds, {len(FEATURES)} features")


In [ ]:
# Train/test split (80/20), fixed seed for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)


In [ ]:
# 5-fold cross-validation on the training set (checks the model isn't just lucky on one split)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="r2")
print("5-fold CV R^2 scores:", np.round(cv_scores, 3))
print(f"Mean CV R^2: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")


In [ ]:
# Held-out test set performance (the real generalization check)
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
mae = mean_absolute_error(y_test, y_pred)

print(f"Test R^2:   {r2:.3f}")
print(f"Test RMSE:  {rmse:.3f} pIC50 units")
print(f"Test MAE:   {mae:.3f} pIC50 units")


In [ ]:
# Plot: predicted vs actual pIC50 on the test set
plt.figure(figsize=(5,5))
plt.scatter(y_test, y_pred, alpha=0.5)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, 'r--', linewidth=1)
plt.xlabel("Actual pIC50")
plt.ylabel("Predicted pIC50")
plt.title("AChE potency model — test set")
plt.tight_layout()
plt.savefig("test_set_performance.png", dpi=150)
plt.show()


In [ ]:
# Feature importance (which descriptors the model actually relies on)
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
plt.figure(figsize=(6,4))
importances.plot(kind="barh")
plt.xlabel("Importance")
plt.title("Feature importance")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150)
plt.show()


In [ ]:
# Save the trained model and the exact feature order the Streamlit app must use
joblib.dump(model, "model.pkl")
with open("feature_names.json", "w") as f:
    json.dump(FEATURES, f)

print("Saved model.pkl and feature_names.json")


In [ ]:
from google.colab import files
files.download("model.pkl")
files.download("feature_names.json")


**Reading the validation numbers honestly:** an R² above ~0.5–0.6 on a held-out test set is a reasonable QSAR result for a single-target IC50 dataset of this size (AChE has a large, well-populated ChEMBL dataset, so this is a favorable case); below that, the model is telling you it needs more data or better descriptors (e.g. Morgan fingerprints instead of, or alongside, the physicochemical descriptors used here) before you'd trust its ranking of compounds.

**Next step:** copy `model.pkl` and `feature_names.json` into the same folder as `app.py`, push everything to GitHub, and deploy on Streamlit Community Cloud (see `README.md`).